In [ ]:
import sys
import polars as pl
import polars.selectors as cs
import pyarrow as pa
import lance
import s3fs
import farmhash
from tqdm.auto import tqdm



In [ ]:
#!{sys.executable} -m pip install -q s3fs pyarrow polars pyfarmhash lance

sys.path.insert(0, '/home/ubuntu/vastaiFinal')


BUCKET = 'solana-flash-chunks-668889064578-ca-central-1-an'
PREFIX = 'project_chunks'
OUT    = '/home/ubuntu/vastaiFinal/processed.lance'

print(f'Source : s3://{BUCKET}/{PREFIX}/*.feather')
print(f'Output : {OUT}')

Source : s3://solana-flash-chunks-668889064578-ca-central-1-an/project_chunks/*.feather
Output : /home/ubuntu/vastaiFinal/processed.lance


In [ ]:
# ── List feather files on S3, read with Polars, cast numerics → float32 ───────
s3 = s3fs.S3FileSystem()
paths = sorted(s3.glob(f'{BUCKET}/{PREFIX}/*.feather'))
print(f'Files found: {len(paths)}')

frames = []
for path in tqdm(paths):
    with s3.open(path, 'rb') as f:
        chunk = pl.read_ipc(f)
    chunk = chunk.with_columns(cs.numeric().cast(pl.Float32))
    # ── Deterministic wallet sample (mirrors BigQuery FARM_FINGERPRINT % 300, 3) ──────
    chunk = chunk.filter(
        pl.col('wallet').map_elements(
            lambda w: abs(farmhash.fingerprint64(w)) % 300 == 0,
            return_dtype=pl.Boolean,
        )
    )
    frames.append(chunk)

df = pl.concat(frames)
del frames

print(f'Rows    : {df.shape[0]:,}')
print(f'Wallets : {df["wallet"].n_unique():,}')
print(f'Columns : {df.columns}')

Files found: 153


  0%|          | 0/153 [00:00<?, ?it/s]

Rows    : 2,905,990
Wallets : 619,160
Columns : ['signature', 'block_timestamp', 'tx_date', 'wallet', 'fee_sol', 'compute_units_consumed', 'success_flag', 'involves_sysvar_flag', 'max_cpi_depth', 'inner_instruction_count', 'dex_hop_count', 'debt_hop_count', 'instigator_net_token_profit', 'instigator_sol_delta', 'instigator_fiat_delta', 'unique_nonsigner_account_count', 'log_count', 'num_accounts', 'num_balance_changes', 'max_sol_change', 'mint_diversity', 'unique_program_count']


In [ ]:
# ── Engineer features (Polars equivalent of preprocess.engineer_features) ─────
exprs = []

# Fill nulls from BigQuery left/outer joins
for col in ('instigator_net_token_profit', 'instigator_fiat_delta', 'instigator_sol_delta'):
    if col in df.columns:
        exprs.append(pl.col(col).fill_null(0.0))

# Composability ratios
if {'unique_program_count', 'unique_nonsigner_account_count'}.issubset(df.columns):
    exprs.append(
        pl.when(pl.col('unique_nonsigner_account_count') == 0)
          .then(0.0)
          .otherwise(pl.col('unique_program_count') / pl.col('unique_nonsigner_account_count'))
          .cast(pl.Float32)
          .alias('hop_density')
    )

if {'max_cpi_depth', 'unique_program_count'}.issubset(df.columns):
    exprs.append(
        pl.when(pl.col('unique_program_count') == 0)
          .then(0.0)
          .otherwise(pl.col('max_cpi_depth') / pl.col('unique_program_count'))
          .cast(pl.Float32)
          .alias('avg_depth_per_protocol')
    )

# Novel contract count
if {'unique_program_count', 'dex_hop_count', 'debt_hop_count'}.issubset(df.columns):
    exprs.append(
        (pl.col('unique_program_count') - pl.col('dex_hop_count') - pl.col('debt_hop_count'))
          .clip(lower_bound=0)
          .cast(pl.Float32)
          .alias('unknown_program_count')
    )

# Profit flags
for col, alias in [
    ('instigator_fiat_delta',        'has_fiat_profit'),
    ('instigator_net_token_profit',  'has_token_profit'),
    ('instigator_sol_delta',         'has_sol_profit'),
]:
    if col in df.columns:
        exprs.append((pl.col(col) > 0).cast(pl.Int8).alias(alias))

df = df.with_columns(exprs)

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns}')

Shape  : (2905990, 28)
Columns: ['signature', 'block_timestamp', 'tx_date', 'wallet', 'fee_sol', 'compute_units_consumed', 'success_flag', 'involves_sysvar_flag', 'max_cpi_depth', 'inner_instruction_count', 'dex_hop_count', 'debt_hop_count', 'instigator_net_token_profit', 'instigator_sol_delta', 'instigator_fiat_delta', 'unique_nonsigner_account_count', 'log_count', 'num_accounts', 'num_balance_changes', 'max_sol_change', 'mint_diversity', 'unique_program_count', 'hop_density', 'avg_depth_per_protocol', 'unknown_program_count', 'has_fiat_profit', 'has_token_profit', 'has_sol_profit']


In [ ]:
cache_path = 'engineered_features_cache.parquet'
# Save the DataFrame to disk
df.write_parquet(cache_path)

print(f'Successfully cached DataFrame to {cache_path}')
print(f'File size: {__import__("os").path.getsize(cache_path) / (1024*1024):.2f} MB')


Successfully cached DataFrame to engineered_features_cache.parquet
File size: 271.45 MB


In [ ]:

cache_path = 'engineered_features_cache.parquet'

pl.read_parquet(cache_path)

signature,block_timestamp,tx_date,wallet,fee_sol,compute_units_consumed,success_flag,involves_sysvar_flag,max_cpi_depth,inner_instruction_count,dex_hop_count,debt_hop_count,instigator_net_token_profit,instigator_sol_delta,instigator_fiat_delta,unique_nonsigner_account_count,log_count,num_accounts,num_balance_changes,max_sol_change,mint_diversity,unique_program_count,hop_density,avg_depth_per_protocol,unknown_program_count,has_fiat_profit,has_token_profit,has_sol_profit
str,"datetime[μs, UTC]",date,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i8
"""4EBGdMPx2wn5A1BFsYPJKnwJtM612K…",2024-09-01 09:53:15 UTC,2024-09-01,"""GBuTZfT1ULsbdknmyJGJKiauAQ6wjy…",0.000105,40966.0,1.0,0.0,2.0,11.0,0.0,0.0,0.0,-0.004184,0.0,6.0,44.0,10.0,10.0,0.004184,0.0,4.0,0.666667,0.5,4.0,0,null,0
"""xv5gXqPPvDjqycAomi1VwE16cLThaU…",2024-09-01 03:03:46 UTC,2024-09-01,"""5y5jCvdD6uTPjWbQtLj8ABQ56jCor2…",0.000017,26756.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002056,0.0,7.0,32.0,11.0,11.0,0.202039,1.0,5.0,0.714286,0.4,5.0,0,null,0
"""3R1oye8fqo7vZCWpB8DmFXJmsLDkSZ…",2024-09-01 22:24:21 UTC,2024-09-01,"""HVVpe8FEKFTA8XzFGyyiJRwn9GwzSo…",0.000015,27854.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002054,-100.0,7.0,32.0,11.0,11.0,0.002054,1.0,5.0,0.714286,0.4,5.0,0,null,0
"""3oB21Z2VmF8s6JXWXjdbU2BvSy4Z8C…",2024-09-01 19:31:12 UTC,2024-09-01,"""G1sd2FVtvwhtowoxLFoVeMcrCkYqnh…",0.000015,29195.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002054,0.0,7.0,32.0,11.0,11.0,0.002054,1.0,5.0,0.714286,0.4,5.0,0,null,0
"""3MwBVvQCKc46STAxity7GoqmtS5PCG…",2024-09-01 03:34:55 UTC,2024-09-01,"""CpHrckWuj3jfXG36bKDr3zJgjNu4D6…",0.000015,27854.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002054,0.0,7.0,32.0,11.0,11.0,0.002054,1.0,5.0,0.714286,0.4,5.0,0,null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""5q9CVRYkkHRL8VuFTvuxNtqxKDmw1x…",2025-01-31 13:13:03 UTC,2025-01-31,"""JARDTp1s5HDNUY4yZsmxEyoVm9JCsj…",0.000006,795825.0,1.0,0.0,null,0.0,0.0,0.0,19999.982422,0.0574,19999.982422,12.0,1.0,15.0,15.0,0.057406,2.0,0.0,0.0,0.0,0.0,1,1,1
"""3FZLeMhV8ZSroMzRnb1hFxZRpBfrTb…",2025-01-31 12:20:58 UTC,2025-01-31,"""JARDTp1s5HDNUY4yZsmxEyoVm9JCsj…",0.000006,815028.0,1.0,0.0,null,0.0,0.0,0.0,20005.066406,0.0574,20005.066406,12.0,1.0,15.0,15.0,0.057406,2.0,0.0,0.0,0.0,0.0,1,1,1
"""5JZiDngyVzyqr2VtQWqS3ZFbBNFMgL…",2025-01-31 13:43:46 UTC,2025-01-31,"""JARDTp1s5HDNUY4yZsmxEyoVm9JCsj…",0.00006,902441.0,1.0,0.0,null,0.0,0.0,0.0,47.844093,2.888813,0.0,13.0,1.0,17.0,17.0,2.888813,2.0,0.0,0.0,0.0,0.0,0,1,1


In [ ]:
# ── Null counts ────────────────────────────────────────────────────────────────
null_df = (
    df.null_count()
      .unpivot(variable_name='column', value_name='null_count')
      .with_columns(
          (pl.col('null_count') / df.shape[0] * 100).round(3).alias('null_pct')
      )
      .sort('null_count', descending=True)
)
print(null_df)

shape: (28, 3)
┌────────────────────────┬────────────┬──────────┐
│ column                 ┆ null_count ┆ null_pct │
│ ---                    ┆ ---        ┆ ---      │
│ str                    ┆ u32        ┆ f64      │
╞════════════════════════╪════════════╪══════════╡
│ has_token_profit       ┆ 5167149    ┆ 16.416   │
│ max_cpi_depth          ┆ 95698      ┆ 0.304    │
│ signature              ┆ 0          ┆ 0.0      │
│ block_timestamp        ┆ 0          ┆ 0.0      │
│ tx_date                ┆ 0          ┆ 0.0      │
│ …                      ┆ …          ┆ …        │
│ hop_density            ┆ 0          ┆ 0.0      │
│ avg_depth_per_protocol ┆ 0          ┆ 0.0      │
│ unknown_program_count  ┆ 0          ┆ 0.0      │
│ has_fiat_profit        ┆ 0          ┆ 0.0      │
│ has_sol_profit         ┆ 0          ┆ 0.0      │
└────────────────────────┴────────────┴──────────┘


In [ ]:
# ── Write Lance ────────────────────────────────────────────────────────────────
table = df.to_arrow()
lance.write_dataset(table, OUT, mode='overwrite')
del table

ds = lance.dataset(OUT)
print(f'Rows   : {ds.count_rows():,}')
print(f'Schema :\n{ds.schema}')